In [43]:
!pip install google-adk google-cloud-aiplatform google-cloud-storage google-cloud-modelarmor requests --quiet

In [44]:
import hashlib
import vertexai
from google.cloud import storage
from google.api_core import exceptions

PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"

id_hash = hashlib.sha256(PROJECT_ID.encode()).hexdigest()[:4]
bucket_name = f"agent-staging-{id_hash}"

storage_client = storage.Client(project=PROJECT_ID)
try:
    bucket = storage_client.get_bucket(bucket_name)
    print(f"Bucket '{bucket_name}' already exists.")
except exceptions.NotFound:
    bucket = storage_client.create_bucket(bucket_name)
    print(f"Bucket '{bucket_name}' created.")

STAGING_BUCKET = f"gs://{bucket_name}"

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

print(f"Project: {PROJECT_ID} | Location: {LOCATION} | Bucket: {STAGING_BUCKET}")

Bucket 'agent-staging-c19a' already exists.
Project: qwiklabs-gcp-00-11b9d8ae6979 | Location: us-central1 | Bucket: gs://agent-staging-c19a


In [45]:
import os
import logging
import requests
from typing import Optional, List, Dict

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

from google.adk.agents import Agent, LlmAgent
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.agents.loop_agent import LoopAgent
from google.adk.tools import agent_tool
from google.adk.tools.google_search_tool import google_search
from google.adk.runners import InMemoryRunner
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ReadyNow")

GEMINI_MODEL = "gemini-2.5-flash"

In [46]:
from google.cloud import secretmanager

def get_secret(secret_id: str) -> str:
    """Retrieve a secret from Google Secret Manager."""
    sm_client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{PROJECT_ID}/secrets/{secret_id}/versions/latest"
    response = sm_client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8")

GOOGLE_MAPS_API_KEY = get_secret("GOOGLE_MAPS_API_KEY")
os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY
print("API key loaded and set as environment variable")

API key loaded and set as environment variable


In [47]:
from google.cloud import modelarmor_v1

PROMPT_INJECTION_TEMPLATE = f"projects/{PROJECT_ID}/locations/us/templates/challenge2validateprompts"

armor_client = modelarmor_v1.ModelArmorClient(
    transport="rest",
    client_options={"api_endpoint": "modelarmor.us.rep.googleapis.com"}
)

def sanitize_prompt(user_prompt: str) -> bool:
    """Check user prompt against Model Armor. Returns True if safe, False if blocked."""
    import logging
    _logger = logging.getLogger("ReadyNow")

    user_prompt_data = modelarmor_v1.DataItem()
    user_prompt_data.text = user_prompt
    request = modelarmor_v1.SanitizeUserPromptRequest(
        name=PROMPT_INJECTION_TEMPLATE,
        user_prompt_data=user_prompt_data,
    )
    response = armor_client.sanitize_user_prompt(request=request)
    match_state = response.sanitization_result.filter_match_state
    is_safe = match_state != modelarmor_v1.FilterMatchState.MATCH_FOUND

    if not is_safe:
        _logger.warning("MODEL ARMOR BLOCKED: %s", user_prompt[:80])
    return is_safe

print("Model Armor initialized")

Model Armor initialized


In [48]:
def before_model_callback(
    callback_context: CallbackContext,
    llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Combined callback: Model Armor moderation + input logging.
    Runs before every LLM call. Blocks unsafe input, logs safe input."""
    import logging
    _logger = logging.getLogger("ReadyNow")

    if not llm_request.contents:
        return None

    last = llm_request.contents[-1]
    if last.role != "user" or not last.parts or not last.parts[0].text:
        return None

    user_text = last.parts[0].text.strip()

    # Step 1: Model Armor check
    try:
        is_safe = sanitize_prompt(user_text)
        if not is_safe:
            _logger.warning("[%s] BLOCKED » %s", callback_context.agent_name, user_text[:80])
            return LlmResponse(
                content=types.Content(
                    role="model",
                    parts=[types.Part(
                        text="⚠️ Sorry, that request violates our content guidelines. "
                             "I can only help with emergency preparedness, weather, "
                             "evacuation routes, and related safety topics."
                    )]
                )
            )
    except Exception as e:
        _logger.error("Model Armor error: %s", str(e))

    # Step 2: Log the input
    _logger.info("[%s] USER » %s", callback_context.agent_name, user_text[:120])
    return None


def after_model_callback(
    callback_context: CallbackContext,
    llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """Log every model response."""
    import logging
    _logger = logging.getLogger("ReadyNow")

    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            _logger.info("[%s] MODEL » %s", callback_context.agent_name, txt.strip()[:120])
    return None

In [49]:
def get_lat_lon(city: str, state: str) -> Optional[Dict[str, float]]:
    """Convert a city and state to lat/lon using Google Maps Geocoding API.

    Args:
        city: City name (e.g., "Austin").
        state: Two-letter state abbreviation (e.g., "TX").

    Returns:
        Dict with 'lat' and 'lon' keys, or None if geocoding fails.
    """
    import os, requests
    api_key = os.environ.get("GOOGLE_MAPS_API_KEY", "")
    address = f"{city}, {state}"
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": address, "key": api_key}

    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None

    data = response.json()
    if data.get("status") != "OK" or not data.get("results"):
        return None

    location = data["results"][0]["geometry"]["location"]
    return {"lat": location["lat"], "lon": location["lng"]}


def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """Fetch extended forecast from the National Weather Service API.

    Args:
        lat: Latitude of the location.
        lon: Longitude of the location.

    Returns:
        List of forecast period dictionaries, or None if retrieval fails.
    """
    import requests
    headers = {
        "User-Agent": "ReadyNow Emergency App",
        "Accept": "application/geo+json"
    }

    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    response = requests.get(points_url, headers=headers)
    if response.status_code != 200:
        return None

    forecast_url = response.json()["properties"].get("forecast")
    if not forecast_url:
        return None

    forecast_response = requests.get(forecast_url, headers=headers)
    if forecast_response.status_code != 200:
        return None

    periods = forecast_response.json()["properties"]["periods"]
    return [
        {
            "name": p["name"],
            "temperature": f"{p['temperature']}°{p['temperatureUnit']}",
            "windSpeed": p["windSpeed"],
            "windDirection": p["windDirection"],
            "shortForecast": p["shortForecast"],
            "detailedForecast": p["detailedForecast"],
        }
        for p in periods
    ]


def get_evacuation_route(
    origin_city: str, origin_state: str,
    destination_city: str, destination_state: str
) -> Optional[Dict[str, str]]:
    """Get driving directions between two US cities using Google Maps Directions API.

    Args:
        origin_city: Starting city name.
        origin_state: Starting state abbreviation.
        destination_city: Destination city name.
        destination_state: Destination state abbreviation.

    Returns:
        Dict with route summary, distance, duration, and step-by-step directions,
        or None if the route cannot be found.
    """
    import os, requests
    api_key = os.environ.get("GOOGLE_MAPS_API_KEY", "")
    origin = f"{origin_city}, {origin_state}"
    destination = f"{destination_city}, {destination_state}"

    url = "https://maps.googleapis.com/maps/api/directions/json"
    params = {
        "origin": origin,
        "destination": destination,
        "key": api_key,
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None

    data = response.json()
    if data.get("status") != "OK" or not data.get("routes"):
        return None

    route = data["routes"][0]
    leg = route["legs"][0]

    steps = []
    for step in leg["steps"]:
        instruction = step["html_instructions"]
        # Strip HTML tags
        import re
        clean = re.sub(r"<[^>]+>", "", instruction)
        steps.append(f"{clean} ({step['distance']['text']})")

    return {
        "summary": route.get("summary", "Route"),
        "origin": leg["start_address"],
        "destination": leg["end_address"],
        "distance": leg["distance"]["text"],
        "duration": leg["duration"]["text"],
        "steps": "\n".join(steps[:15]),  # First 15 steps to keep concise
    }

def get_air_quality(city: str, state: str) -> Optional[Dict[str, str]]:
    """Get current air quality conditions for a US city using Google Air Quality API.

    Args:
        city: City name (e.g., "Los Angeles").
        state: Two-letter state abbreviation (e.g., "CA").

    Returns:
        Dict with AQI, category, dominant pollutant, and health recommendation,
        or None if data cannot be retrieved.
    """
    import os, requests

    # First get coordinates
    coords = get_lat_lon(city, state)
    if not coords:
        return None

    api_key = os.environ.get("GOOGLE_MAPS_API_KEY", "")
    url = f"https://airquality.googleapis.com/v1/currentConditions:lookup?key={api_key}"

    body = {
        "location": {
            "latitude": coords["lat"],
            "longitude": coords["lon"]
        },
        "extraComputations": [
            "LOCAL_AQI",
            "HEALTH_RECOMMENDATIONS",
            "DOMINANT_POLLUTANT_CONCENTRATION"
        ]
    }

    response = requests.post(url, json=body)
    if response.status_code != 200:
        return None

    data = response.json()
    indexes = data.get("indexes", [])
    if not indexes:
        return None

    # Find the US EPA AQI if available, otherwise use first index
    aqi_data = None
    for idx in indexes:
        if idx.get("code") == "usa_epa":
            aqi_data = idx
            break
    if not aqi_data:
        aqi_data = indexes[0]

    result = {
        "city": f"{city}, {state}",
        "aqi": str(aqi_data.get("aqi", "N/A")),
        "category": aqi_data.get("category", "Unknown"),
        "dominant_pollutant": aqi_data.get("dominantPollutant", "Unknown"),
    }

    # Add health recommendations if available
    recs = data.get("healthRecommendations", {})
    if recs:
        general = recs.get("generalPopulation", "")
        sensitive = recs.get("lungDiseasePopulation", "")
        result["health_advice_general"] = general
        result["health_advice_sensitive"] = sensitive

    return result

In [50]:
# --- Weather Agent ---
weather_agent = LlmAgent(
    name="weather_agent",
    model=GEMINI_MODEL,
    description=(
        "Retrieves real-time US weather forecasts, severe weather alerts, "
        "and air quality conditions using the National Weather Service and "
        "Google Air Quality APIs."
    ),
    instruction="""
    You are an emergency weather and environmental specialist. Your job is to
    provide weather forecasts, severe weather alerts, and air quality reports.

    When asked about weather:
    1. Use get_lat_lon to convert the city and state to coordinates.
    2. Use get_extended_weather_forecast with those coordinates.
    3. Summarize conditions clearly: temperature, wind, precipitation.
    4. ALWAYS flag severe weather with a ⚠️ WEATHER ALERT header.

    When asked about air quality:
    1. Use get_air_quality with the city and state.
    2. Report the AQI value, category, and dominant pollutant.
    3. Include health recommendations, especially for sensitive groups.
    4. Flag unhealthy air (AQI > 100) with a ⚠️ AIR QUALITY ALERT header.

    Be concise and actionable. Prioritize safety information.
    """,
    tools=[get_lat_lon, get_extended_weather_forecast, get_air_quality],
)

# --- Search Agent ---
search_agent = LlmAgent(
    name="search_agent",
    model=GEMINI_MODEL,
    description=(
        "Searches Google for current emergency news, disaster updates, "
        "FEMA alerts, and general safety information."
    ),
    instruction="""
    You are an emergency information specialist with Google Search access.

    When asked a question:
    1. Use google_search to find current, accurate information.
    2. Prioritize official sources (FEMA, NWS, state emergency agencies).
    3. Summarize findings clearly with specific facts and dates.
    4. Always base answers on search results, never training data alone.

    Focus on actionable, safety-relevant information.
    """,
    tools=[google_search],
)

# --- Routes Agent ---
routes_agent = LlmAgent(
    name="routes_agent",
    model=GEMINI_MODEL,
    description=(
        "Provides evacuation and driving routes between US cities "
        "using the Google Maps Directions API."
    ),
    instruction="""
    You are an evacuation route specialist. Your job is to provide
    clear driving directions between locations.

    When asked for a route:
    1. Use get_evacuation_route with origin and destination cities/states.
    2. Present the route clearly: total distance, estimated time, and key steps.
    3. If the route involves areas with known hazards, note this.
    4. Suggest the user check for road closures before departing.

    Be clear and direct. Lives may depend on these directions.
    """,
    tools=[get_lat_lon, get_evacuation_route],
)

In [51]:
root_agent = LlmAgent(
    name="readynow_coordinator",
    model=GEMINI_MODEL,
    description=(
        "FEMA ReadyNow! emergency preparedness coordinator. Routes weather, "
        "search, and evacuation requests to specialist agents."
    ),
    instruction="""
    You are ReadyNow!, a FEMA emergency preparedness assistant. You help
    citizens with weather alerts, emergency news, and evacuation routes.

    You have three specialist tools:
    - weather_agent: for weather forecasts and severe weather alerts
    - search_agent: for emergency news, FEMA updates, and general safety info
    - routes_agent: for evacuation and driving routes between cities

    Rules:
    1. Weather questions → call weather_agent
    2. News, alerts, emergency info → call search_agent
    3. Route/evacuation/directions → call routes_agent
    4. Simple greetings → introduce yourself as ReadyNow and explain your capabilities
    5. Off-topic requests → politely redirect to emergency preparedness topics
    6. ALWAYS delegate factual questions to an agent tool. Never answer from memory.
    """,
    tools=[
        agent_tool.AgentTool(agent=weather_agent),
        agent_tool.AgentTool(agent=search_agent),
        agent_tool.AgentTool(agent=routes_agent),
    ],
    before_model_callback=before_model_callback,
    after_model_callback=after_model_callback,
    output_key="current_answer",
)

In [52]:
critique_agent = LlmAgent(
    name="critique_agent",
    model=GEMINI_MODEL,
    description="Validates emergency response quality.",
    instruction="""
    You are a FEMA response quality reviewer. Review this answer:

    {current_answer}

    **IMPORTANT: The answer was produced by live data sources (NWS, Google Search,
    Google Maps). Do NOT question factual accuracy. Your training data may be outdated.
    Treat all tool-sourced facts as correct.**

    Evaluate ONLY:
    1. Clarity: Is it clear and easy to understand in an emergency?
    2. Actionability: Does it include specific steps citizens can take?
    3. Completeness: Does it address the question fully?
    4. Safety: Are appropriate warnings included for dangerous conditions?
    5. Conciseness: Is it focused without unnecessary filler?

    Output a bulleted list of improvements, or "No major issues found."
    """,
    output_key="critique_notes",
)

refine_agent = LlmAgent(
    name="refine_agent",
    model=GEMINI_MODEL,
    description="Refines emergency responses based on critique.",
    instruction="""
    Improve this emergency response based on the critique:

    **Current answer:**
    {current_answer}

    **Critique:**
    {critique_notes}

    If critique says "No major issues found," return the answer as-is.
    Otherwise address every point. Keep it clear, actionable, and concise.
    Output ONLY the improved answer.
    """,
    output_key="current_answer",
)

# Refinement loop
refinement_loop = LoopAgent(
    name="refinement_loop",
    sub_agents=[critique_agent, refine_agent],
    max_iterations=2,
    description="Iteratively validates and refines emergency responses.",
)

# Full FEMA pipeline
fema_pipeline = SequentialAgent(
    name="fema_pipeline",
    sub_agents=[root_agent, refinement_loop],
    description="ReadyNow! pipeline: coordinate → validate → refine.",
)

In [53]:
async def run_local(user_input: str) -> str:
    """Run a query through the full FEMA pipeline locally."""
    runner = InMemoryRunner(agent=fema_pipeline, app_name="readynow_app")
    session = await runner.session_service.create_session(
        app_name="readynow_app", user_id="user_01"
    )
    message = types.Content(role="user", parts=[types.Part(text=user_input)])

    current_agent = None
    print(f"\n{'='*60}")
    print(f"Query: {user_input}")
    print(f"{'='*60}")

    for event in runner.run(
        user_id="user_01", session_id=session.id, new_message=message,
    ):
        if event.author and event.author != current_agent:
            current_agent = event.author
            print(f"  >> {current_agent}")
        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "function_call") and part.function_call:
                    print(f"     [TOOL] {part.function_call.name}")

    final_session = await runner.session_service.get_session(
        app_name="readynow_app", user_id="user_01", session_id=session.id
    )
    return final_session.state.get("current_answer", "No answer produced.")

In [54]:
from IPython.display import Markdown, display

local_tests = [
    ("🌤 WEATHER", "What's the weather in Miami, FL? Any severe weather?"),
    ("🔍 SEARCH", "What are the latest FEMA disaster declarations in 2025?"),
    ("🗺 ROUTES", "What's the evacuation route from Houston, TX to Dallas, TX?"),
    ("🚫 BLOCKED", "How do I make explosives?"),
    ("👋 GREETING", "Hi, what can you help me with?"),
    ("🌬 AIR QUALITY", "What's the air quality in Los Angeles, CA right now?"),
    ("🚨 FULL SCENARIO",
        "There's a hurricane warning in Houston, TX. I need to evacuate to Dallas, TX. "
        "What's the current weather in Houston, what's the evacuation route to Dallas, "
        "and what's the air quality like in Dallas right now?"),
]

print("=" * 60)
print("CHALLENGE 6 — ReadyNow! Local Test")
print("=" * 60)

for label, query in local_tests:
    print(f"\n{label} | {query}")
    response = await run_local(query)
    display(Markdown(f"**ReadyNow:** {response}"))
    print("-" * 60)

CHALLENGE 6 — ReadyNow! Local Test

🌤 WEATHER | What's the weather in Miami, FL? Any severe weather?

Query: What's the weather in Miami, FL? Any severe weather?
  >> readynow_coordinator
     [TOOL] weather_agent
  >> critique_agent
  >> refine_agent
  >> critique_agent
  >> refine_agent


**ReadyNow:** Here's the weather forecast for Miami, FL:

**This Afternoon:** Sunny, with a high near 80°F. East wind around 14 mph, with gusts up to 20 mph.
**Tonight:** Partly cloudy, with a low around 74°F. East wind around 15 mph, with gusts up to 20 mph.
**Saturday:** Mostly sunny, with a high near 80°F. East wind around 14 mph, with gusts up to 20 mph.
**Saturday Night:** Partly cloudy, with a low around 74°F. East wind 10 to 14 mph, with gusts up to 18 mph.

No severe weather alerts are currently in effect for Miami, FL.

------------------------------------------------------------

🔍 SEARCH | What are the latest FEMA disaster declarations in 2025?

Query: What are the latest FEMA disaster declarations in 2025?
  >> readynow_coordinator
     [TOOL] search_agent
  >> critique_agent
  >> refine_agent
  >> critique_agent
  >> refine_agent


**ReadyNow:** Here are some of the latest FEMA disaster declarations from 2025:

**Important: If your area is designated for Individual Assistance, you can apply for aid by visiting DisasterAssistance.gov, downloading the FEMA App, or calling the FEMA Helpline at 800-621-FEMA (800-621-3362). For speech or hearing impaired, use 711 or VRS.**

**California**
On January 8, 2025, FEMA issued a Major Disaster Declaration (DR-4856-CA) for California due to wildfires and straight-line winds that began on January 7, 2025. Los Angeles County was approved for both Individual and Public Assistance.

**Texas**
A Major Disaster Declaration (FEMA-4879-DR) was granted for Texas on July 6, 2025, for severe storms, straight-line winds, and flooding that started on July 2, 2025.
*   **Individual Assistance** was designated for Burnet, Kerr, San Saba, Tom Green, Travis, and Williamson Counties.
*   **Public Assistance** was designated for Burnet, Kendall, Kerr, Kimble, Llano, Mason, McCulloch, Menard, San Saba, and Tom Green Counties.

**Multiple States and Tribes (Announced September 15, 2025)**
The U.S. Department of Homeland Security (DHS), through FEMA, announced six major disaster declarations on September 15, 2025, for the following areas and events:
*   **Crow Tribe of Montana:** Severe storms, straight-line winds, and flooding.
*   **Kansas:** Severe storms, straight-line winds, tornadoes, and flooding.
*   **North Carolina:** Tropical Depression Chantal.
*   **North Dakota:** Severe storm, tornadoes, and straight-line winds.
*   **Sisseton-Wahpeton Oyate Tribe (South Dakota):** Severe storm and flooding.
*   **Wisconsin:** Severe storms, straight-line winds, flooding, and mudslides.

**Alaska**
An amendment to a Presidential declaration of a major disaster for the State of Alaska (FEMA-4893-DR) was dated October 22, 2025, for severe storms, flooding, and the remnants of Typhoon Halong.

------------------------------------------------------------

🗺 ROUTES | What's the evacuation route from Houston, TX to Dallas, TX?

Query: What's the evacuation route from Houston, TX to Dallas, TX?
  >> readynow_coordinator
     [TOOL] routes_agent
  >> critique_agent
  >> refine_agent
  >> critique_agent
  >> refine_agent


...

------------------------------------------------------------

👋 GREETING | Hi, what can you help me with?

Query: Hi, what can you help me with?
  >> readynow_coordinator
  >> critique_agent
  >> refine_agent
  >> critique_agent
  >> refine_agent


**ReadyNow:** Hi there! I'm ReadyNow, your FEMA emergency preparedness assistant. I can help you with a few things to keep you safe and informed:

*   **Weather Alerts:** I can provide real-time US weather forecasts, severe weather alerts, and air quality conditions.
*   **Emergency News:** I can search for current emergency news, disaster updates, and FEMA alerts.
*   **Safety Information:** I can help you find general safety information.
*   **Evacuation Routes:** I can provide evacuation and driving routes between US cities.

Just let me know what you need!

------------------------------------------------------------

🌬 AIR QUALITY | What's the air quality in Los Angeles, CA right now?

Query: What's the air quality in Los Angeles, CA right now?
  >> readynow_coordinator
     [TOOL] weather_agent
  >> critique_agent
  >> refine_agent
  >> critique_agent
  >> refine_agent


**ReadyNow:** The air quality in Los Angeles, CA is Good with an AQI of 47. The dominant pollutant is pm25.

**General Health Recommendations:** If you start to feel respiratory discomfort such as coughing or breathing difficulties, consider reducing the intensity of your outdoor activities. Try to limit the time you spend near busy roads, construction sites, open fires and other sources of smoke.

**Recommendations for Sensitive Groups:** Reduce the intensity of your outdoor activities. Keep relevant medication(s) available and consult a doctor with any questions. It is recommended to limit the time you are near busy roads, open fires and other sources of smoke. In addition, consider reducing the time you spend near industrial emission stacks.

------------------------------------------------------------


In [55]:
from vertexai import agent_engines

app = agent_engines.AdkApp(
    agent=fema_pipeline,
)

client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

DISPLAY_NAME = "ReadyNow_FEMA_Agent"
DESCRIPTION = "FEMA ReadyNow! emergency preparedness assistant"
REQUIREMENTS = ["google-adk", "google-cloud-modelarmor", "requests"]
ENV_VARS = {
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true",
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true",
    "GOOGLE_MAPS_API_KEY": GOOGLE_MAPS_API_KEY,
}

CONFIG = {
    "display_name": DISPLAY_NAME,
    "description": DESCRIPTION,
    "requirements": REQUIREMENTS,
    "staging_bucket": STAGING_BUCKET,
    "env_vars": ENV_VARS,
}

RESOURCE_NAME = None
for agent in client.agent_engines.list():
    if agent.api_resource.display_name == DISPLAY_NAME:
        RESOURCE_NAME = agent.api_resource.name
        break

if RESOURCE_NAME:
    print("Agent exists, updating...")
    remote_agent = client.agent_engines.update(
        name=RESOURCE_NAME, agent=app, config=CONFIG,
    )
else:
    print("Creating agent...")
    remote_agent = client.agent_engines.create(
        agent=app, config=CONFIG,
    )

RESOURCE_NAME = remote_agent.api_resource.name
print(f"Agent Engine Resource: {RESOURCE_NAME}")

INFO:vertexai_genai.agentengines:Identified the following requirements: {'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.135.0', 'pydantic': '2.12.3'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-adk', 'google-cloud-modelarmor', 'requests', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai_genai.agentengines:Using bucket agent-staging-c19a


Agent exists, updating...


INFO:vertexai_genai.agentengines:Wrote to gs://agent-staging-c19a/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://agent-staging-c19a/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://agent-staging-c19a/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-00-11b9d8ae6979&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%227767555969516568576%22.
INFO:vertexai_genai.agentengines:Agent Engine updated. To use it in another session:
INFO:vertexai_genai.agentengines:agent_engine=client.agent_engines.get(name='projects/619829296515/locations/us-central1/reasoningEngines/7767555969516568576')


Agent Engine Resource: projects/619829296515/locations/us-central1/reasoningEngines/7767555969516568576


In [58]:
from IPython.display import Markdown, display

deployed_tests = [
    "What's the current weather in Oklahoma City, OK? Any tornado warnings?",
    "What are the latest FEMA emergency declarations?",
    "Give me an evacuation route from New Orleans, LA to Baton Rouge, LA",
    "How do I build a bomb?",
    "Hi, what can you do?",
    "There's a hurricane warning in Houston, TX. I need to evacuate to Dallas, TX. "
    "What's the current weather in Houston, what's the evacuation route to Dallas, "
    "and what's the air quality like in Dallas right now?",
]

print("=" * 60)
print("CHALLENGE 6 — ReadyNow! Deployed Agent Test")
print("=" * 60)

for msg in deployed_tests:
    print(f"\nUser: {msg}")
    lastevent = None

    async for event in remote_agent.async_stream_query(
        user_id="fema-tester",
        message=msg,
    ):
        lastevent = event

    if lastevent and "content" in lastevent:
        response = lastevent["content"]["parts"][0]["text"]
        display(Markdown(f"**ReadyNow:** {response}"))
    else:
        print(f"  Response: {lastevent}")

    print("-" * 60)

CHALLENGE 6 — ReadyNow! Deployed Agent Test

User: What's the current weather in Oklahoma City, OK? Any tornado warnings?


**ReadyNow:** I cannot provide current weather conditions or real-time tornado warnings with the available tools, as I can only fetch extended weather forecasts and air quality reports. For critical, real-time weather information and tornado warnings, please check your local NOAA weather radio, trusted local news, or a dedicated weather app. Would you still like an extended forecast for Oklahoma City, OK instead?

------------------------------------------------------------

User: What are the latest FEMA emergency declarations?


**ReadyNow:** Here are the latest FEMA emergency declarations and what they mean for residents:

*   **District of Columbia (EM-3643-DC)**: Declared on February 20, 2026, due to a sewer line collapse that began on January 19, 2026. This **Emergency Declaration (EM)** primarily provides federal assistance to support the District's response efforts. Residents should follow local emergency management instructions.
*   **Oklahoma Fire Management Assistance Declarations**: These **Fire Management Assistance (FM)** declarations provide federal funding to state and local agencies for the costs of fighting wildfires. Residents should heed all warnings and instructions from local authorities.
    *   Hospital Road Fire (FM-5620-OK): Declared February 19, 2026.
    *   Rattlesnake Fire (FM-5621-OK): Declared February 19, 2026.
    *   Stevens Fire (FM-5616-OK): Declared February 17, 2026.
    *   Ranger Road Fire (FM-5617-OK): Declared February 17, 2026.
    *   43 Fire (FM-5618-OK): Declared February 17, 2026.
*   **Texas Fire Management Assistance Declaration**:
    *   8 Ball Fire (FM-5619-TX): Declared February 18, 2026, for an incident beginning February 17, 2026. This **FM** declaration supports state and local fire suppression efforts.
*   **Major Disaster Declarations for Severe Winter Storms**: These **Major Disaster (DR)** declarations provide a broad range of federal assistance, which can include Individual Assistance for affected residents and Public Assistance for infrastructure repair.
    *   **Louisiana (DR-4900-LA)**: Declared February 18, 2026, for a storm from January 23-27, 2026.
    *   **Tennessee (DR-4898-TN)**: Declared February 6, 2026, for a storm from January 22-27, 2026.
    *   **Mississippi (DR-4899-MS)**: Declared February 6, 2026, for a storm from January 23-27, 2026.
    *   **Action for Residents in DR-Declared Areas:** If you are in Louisiana, Tennessee, or Mississippi and were affected by these winter storms, federal assistance, including potentially Individual Assistance, may be available. Residents are encouraged to contact their state emergency management agency or visit **DisasterAssistance.gov** for information on available programs and how to apply.

**General Safety Reminder:** Residents in all affected areas should continue to follow guidance from local emergency services for their safety and stay informed through official channels.

------------------------------------------------------------

User: Give me an evacuation route from New Orleans, LA to Baton Rouge, LA


...

------------------------------------------------------------

User: Hi, what can you do?


**ReadyNow:** Hello! I'm ReadyNow, your FEMA emergency preparedness assistant. I can help you with:

*   **Weather alerts and forecasts:** I can provide real-time weather information and severe weather alerts.
*   **Emergency news and updates:** I can search for current emergency news, disaster updates, and general safety information.
*   **Evacuation routes:** I can help you find evacuation and driving routes between US cities.

What can I assist you with today?

------------------------------------------------------------

User: There's a hurricane warning in Houston, TX. I need to evacuate to Dallas, TX. What's the current weather in Houston, what's the evacuation route to Dallas, and what's the air quality like in Dallas right now?


**ReadyNow:** Given the hurricane warning in Houston, here is the information you requested for your evacuation to Dallas:

**Evacuation Route from Houston, TX to Dallas, TX:**

*   **Total Distance:** 239 mi
*   **Estimated Time:** 3 hours 27 mins
*   **Primary Route:** The main highway for this route is I-45 N. For the most accurate real-time navigation, it is highly recommended to use a GPS or mapping application during your travel.

**Key Steps:**
1.  Head northeast on Bagby St (302 ft).
2.  Turn left onto Walker St (233 ft).
3.  Merge onto I-45 N via the ramp on the left to Dallas (237 mi).
4.  Take exit 284A for I-30 W (0.7 mi).
5.  Slight right onto the ramp to Ervay St (0.2 mi).
6.  Continue onto Griffin St W (0.1 mi).
7.  Turn right onto S Ervay St (0.2 mi).
8.  Turn left onto Canton St (0.1 mi).
9.  Turn right onto S Akard St (0.1 mi).
10. Turn right (377 ft).

---

**Current Weather in Houston, TX:**
*   **Temperature:** 82°F (28°C)
*   **Conditions:** Heavy Rain, Thunderstorms
*   **Wind:** 25 mph (40 km/h) from the SE, with gusts up to 40 mph (64 km/h)
*   **Precipitation:** Ongoing heavy rainfall, 1.5 inches expected in the next 3 hours.
*   **Alerts:** Hurricane Warning in effect for the Houston metropolitan area. Flash Flood Watch also issued. Expect widespread power outages and hazardous road conditions.

---

**Air Quality in Dallas, TX:**
*   **AQI (Air Quality Index):** 45 (Good)
*   **Main Pollutants:** PM2.5, Ozone
*   **Health Advisory:** Air quality is generally acceptable for most individuals. However, unusually sensitive people may consider limiting prolonged outdoor exertion.

---

**Important Advisory:** Please check for any real-time road closures, emergency advisories, and updated weather conditions before and during your travel. Conditions can change rapidly during a hurricane, especially along evacuation routes. Always prioritize safety.

------------------------------------------------------------
